# Nor-Lea Hospital — Housing Affordability Map
**Lovington, New Mexico**

This notebook builds an interactive map showing which census tracts within a 2-hour drive of Nor-Lea Hospital are affordable for a median registered nurse salary.

Run cells top to bottom. The final cell saves `nor_lea_affordability.html`.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run this once. You can skip it if these are already installed.
import sys
!{sys.executable} -m pip install geopandas pandas requests folium shapely python-dotenv flexpolyline branca --quiet

In [ ]:
# ── Cell 2: API Keys ──────────────────────────────────────────────────────────
# Paste your keys directly here.

CENSUS_API_KEY = "7896a3dae4aea2f40ca7e1d2c51d4ba599b0df8e"
HERE_API_KEY   = "NMgWhRrJX-g0RKz3Fo2mPSHm2VfDLI_PdbVi2gacGsk"

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────

# Salary & affordability
SALARY_ANNUAL    = 88_260
MONTHLY_INCOME   = SALARY_ANNUAL / 12
AFFORD_THRESHOLD = MONTHLY_INCOME * 0.30   # $2,207.50/mo (30% rule)

# Hospital
HOSPITAL_NAME = "Nor-Lea Hospital"
HOSPITAL_LAT  = 32.98049203086512
HOSPITAL_LON  = -103.34531372633968

# Drive-time bands (minutes)
DRIVE_TIMES = [10, 20, 30, 60, 120]

# Census
CENSUS_YEAR = 2024
STATE_FIPS  = "35"    # New Mexico
COUNTY_FIPS = "025"   # Lea County

# ACS variable: B25064_001E = median gross rent, all units
RENT_VAR_CODE = "B25064_001E"
RENT_COL      = "median_rent"
CENSUS_NULL   = -666_666_666

# Map colors
COLOR_NO_DATA    = "#cccccc"
GRADIENT_COLORS  = ["#1b4332", "#52b788", "#b7e4c7", "#f9c74f", "#d62828"]
RING_COLORS      = {10: "#c0392b", 20: "#e67e22", 30: "#f1c40f", 60: "#27ae60", 120: "#2980b9"}

OUTPUT_HTML = "nor_lea_affordability.html"

print(f"Affordability threshold: ${AFFORD_THRESHOLD:,.2f}/mo")

In [ ]:
# ── Cell 4: Imports ───────────────────────────────────────────────────────────

import io
import requests
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
import flexpolyline as fp
from shapely.geometry import Polygon

print("All imports successful.")

In [ ]:
# ── Cell 5: Fetch ACS Rent Data ───────────────────────────────────────────────
print("Fetching ACS rent data from Census API...")

base_url = f"https://api.census.gov/data/{CENSUS_YEAR}/acs/acs5"
params = {
    "get":  f"NAME,{RENT_VAR_CODE}",
    "for":  "tract:*",
    "in":   f"state:{STATE_FIPS}",
    "key":  CENSUS_API_KEY,
}

resp = requests.get(base_url, params=params, timeout=60)
resp.raise_for_status()

data = resp.json()
rent_df = pd.DataFrame(data[1:], columns=data[0])
rent_df = rent_df.rename(columns={RENT_VAR_CODE: RENT_COL})

# Build 11-character GEOID
rent_df["GEOID"] = (
    rent_df["state"].str.zfill(2)
    + rent_df["county"].str.zfill(3)
    + rent_df["tract"].str.zfill(6)
)

# Clean rent values
rent_df[RENT_COL] = pd.to_numeric(rent_df[RENT_COL], errors="coerce")
rent_df[RENT_COL] = rent_df[RENT_COL].where(
    (rent_df[RENT_COL] != CENSUS_NULL) & (rent_df[RENT_COL] >= 0), other=None
)

# Affordability flag
rent_df["affordable"] = rent_df[RENT_COL].apply(
    lambda r: (r <= AFFORD_THRESHOLD) if pd.notna(r) else None
)

rent_df = rent_df[["GEOID", "NAME", RENT_COL, "affordable"]].reset_index(drop=True)

has_data = rent_df[RENT_COL].notna().sum()
print(f"  {len(rent_df):,} tracts retrieved, {has_data:,} with rent data.")
rent_df[rent_df["GEOID"].str.startswith(STATE_FIPS + COUNTY_FIPS)].head()

In [ ]:
# ── Cell 6: Download TIGER/Line Tract Shapefile ───────────────────────────────
print("Downloading TIGER/Line census tract shapefile...")

tiger_url = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_35_tract.zip"
resp = requests.get(tiger_url, timeout=120)
resp.raise_for_status()

tracts_gdf = gpd.read_file(io.BytesIO(resp.content))
tracts_gdf["GEOID"] = tracts_gdf["GEOID"].astype(str)
if tracts_gdf.crs is None or tracts_gdf.crs.to_epsg() != 4326:
    tracts_gdf = tracts_gdf.to_crs(epsg=4326)

print(f"  {len(tracts_gdf):,} tract geometries loaded.")

In [ ]:
# ── Cell 7: Merge Rent Data onto Tract Geometries ─────────────────────────────
print("Merging rent data onto tract geometries...")

merged_gdf = tracts_gdf.merge(rent_df, on="GEOID", how="left")
matched = merged_gdf[RENT_COL].notna().sum()
print(f"  {matched:,} tracts matched with rent data.")

In [ ]:
# ── Cell 8: Fetch Drive-Time Isochrones from HERE Maps ────────────────────────
print("Fetching drive-time isochrones from HERE Maps...")

HERE_ENDPOINT = "https://isoline.router.hereapi.com/v8/isolines"
iso_rows = []

for minutes in DRIVE_TIMES:
    params = {
        "transportMode": "car",
        "origin":        f"{HOSPITAL_LAT},{HOSPITAL_LON}",
        "range[type]":   "time",
        "range[values]": str(minutes * 60),
        "routingMode":   "fast",
        "apikey":        HERE_API_KEY,
    }
    r = requests.get(HERE_ENDPOINT, params=params, timeout=60)
    if not r.ok:
        raise RuntimeError(f"HERE API error {r.status_code} for {minutes}-min: {r.text}")

    polygons = r.json()["isolines"][0]["polygons"]
    rings = [fp.decode(p["outer"]) for p in polygons]
    coords_latlon = max(rings, key=len)
    coords_lonlat = [(lon, lat) for lat, lon, *_ in coords_latlon]
    iso_rows.append({"drive_minutes": minutes, "geometry": Polygon(coords_lonlat)})
    print(f"  {minutes}-min isochrone fetched.")

isochrones_gdf = gpd.GeoDataFrame(iso_rows, crs="EPSG:4326")
isochrones_gdf = isochrones_gdf.sort_values("drive_minutes", ascending=False).reset_index(drop=True)
print("Done.")

In [ ]:
# ── Cell 9: Spatial Join — Assign Tracts to Drive-Time Bands ─────────────────
print("Joining census tracts to isochrone rings...")

iso_cols = isochrones_gdf[["drive_minutes", "geometry"]].copy()

# Intersects join (captures large rural tracts that overlap a ring boundary)
joined = gpd.sjoin(
    merged_gdf[["GEOID", "geometry"]],
    iso_cols,
    how="left",
    predicate="intersects",
)

# Keep the smallest (innermost) drive_minutes per tract
joined = joined.sort_values("drive_minutes", ascending=True)
joined = joined.drop_duplicates(subset="GEOID", keep="first")

# Merge drive_minutes back onto full GeoDataFrame
joined_gdf = merged_gdf.copy()
drive_map = joined.set_index("GEOID")["drive_minutes"]
joined_gdf["drive_minutes"] = joined_gdf["GEOID"].map(drive_map)

within_120 = joined_gdf[joined_gdf["drive_minutes"].notna()]
print(f"  {len(within_120):,} NM tracts intersect the 120-min isochrone.")

In [ ]:
# ── Cell 10: Print Affordability Summary ─────────────────────────────────────

print("=" * 62)
print("  AFFORDABILITY SUMMARY — Nor-Lea Hospital, Lovington NM")
print(f"  RN salary ${SALARY_ANNUAL:,}/yr · threshold ${AFFORD_THRESHOLD:,.2f}/mo")
print("=" * 62)

for minutes in sorted(DRIVE_TIMES):
    band     = joined_gdf[joined_gdf["drive_minutes"] == minutes]
    has_rent = band[RENT_COL].notna().sum()
    n_aff    = band["affordable"].eq(True).sum()
    n_not    = band["affordable"].eq(False).sum()
    pct      = (n_aff / has_rent * 100) if has_rent else 0
    print(f"\n  Within {minutes}-min drive ({len(band)} tracts)")
    print(f"    Affordable: {n_aff}/{has_rent} with data ({pct:.0f}%)  |  Not affordable: {n_not}")

lea = joined_gdf[joined_gdf["GEOID"].str.startswith(STATE_FIPS + COUNTY_FIPS)]
lea_has = lea[RENT_COL].notna().sum()
lea_aff = lea["affordable"].eq(True).sum()
print(f"\n  Lea County ({len(lea)} tracts): {lea_aff}/{lea_has} affordable ({lea_aff/lea_has*100:.0f}%)")

In [ ]:
# ── Cell 11: Build Interactive Map ────────────────────────────────────────────
print("Building map...")

within_120 = joined_gdf[joined_gdf["drive_minutes"].notna()].copy()

# County boundaries: dissolve all NM tracts by county FIPS
joined_gdf["_county"] = joined_gdf["GEOID"].str[:5]
county_gdf = joined_gdf.dissolve(by="_county").reset_index()

# Colormap: dark green (low rent) → red (high/above threshold)
valid_rents = within_120[RENT_COL].dropna()
min_rent  = max(0.0, float(valid_rents.min()))
scale_max = max(AFFORD_THRESHOLD * 1.1, float(valid_rents.max()) * 1.1)

colormap = cm.LinearColormap(
    colors=GRADIENT_COLORS,
    index=[
        min_rent,
        min_rent + (AFFORD_THRESHOLD - min_rent) * 0.4,
        AFFORD_THRESHOLD * 0.85,
        AFFORD_THRESHOLD,
        scale_max,
    ],
    vmin=min_rent,
    vmax=scale_max,
    caption=f"Median Gross Rent  |  Affordable threshold: ${AFFORD_THRESHOLD:,.0f}/mo",
)

def tract_color(rent):
    if pd.isna(rent):
        return COLOR_NO_DATA
    return colormap(float(rent))

# Base map
fmap = folium.Map(
    location=[HOSPITAL_LAT, HOSPITAL_LON],
    zoom_start=7,
    tiles="CartoDB positron",
    control_scale=True,
)

# Census tract fill layer
tract_group = folium.FeatureGroup(name="Census Tracts (rent affordability)", show=True)
for _, row in within_120.iterrows():
    rent  = row[RENT_COL]
    fill  = tract_color(rent)
    drive = f"{int(row['drive_minutes'])} min" if pd.notna(row["drive_minutes"]) else "—"
    rent_str   = f"${rent:,.0f}/mo" if pd.notna(rent) else "No data"
    afford_str = ("Yes" if pd.notna(rent) and rent <= AFFORD_THRESHOLD
                  else ("No" if pd.notna(rent) else "No data"))
    tooltip = (
        f"<b>{row.get('NAME', row['GEOID'])}</b><br>"
        f"Median rent: {rent_str}<br>"
        f"Affordable for RN: {afford_str}<br>"
        f"Drive time: {drive}"
    )
    folium.GeoJson(
        data=row["geometry"].__geo_interface__,
        style_function=lambda _, f=fill: {
            "fillColor": f, "color": "white", "weight": 0.4, "fillOpacity": 0.75
        },
        tooltip=folium.Tooltip(tooltip),
    ).add_to(tract_group)
tract_group.add_to(fmap)

# Tract boundary lines
tract_border_group = folium.FeatureGroup(name="Tract boundaries", show=True)
folium.GeoJson(
    data=within_120.__geo_interface__,
    style_function=lambda _: {"color": "#666666", "weight": 0.6, "fillOpacity": 0.0},
).add_to(tract_border_group)
tract_border_group.add_to(fmap)

# County boundary lines
county_group = folium.FeatureGroup(name="County boundaries", show=True)
folium.GeoJson(
    data=county_gdf.__geo_interface__,
    style_function=lambda _: {"color": "#333333", "weight": 1.8, "fillOpacity": 0.0},
    tooltip=folium.GeoJsonTooltip(fields=["_county"], aliases=["County FIPS:"]),
).add_to(county_group)
county_group.add_to(fmap)

# Isochrone rings
RING_DASH = "8 4"
for _, iso_row in isochrones_gdf.sort_values("drive_minutes", ascending=False).iterrows():
    minutes = int(iso_row["drive_minutes"])
    color   = RING_COLORS.get(minutes, "#555555")
    folium.GeoJson(
        data=iso_row["geometry"].__geo_interface__,
        name=f"{minutes}-min drive ring",
        style_function=lambda _, c=color: {
            "color": c, "weight": 2.5, "dashArray": RING_DASH, "fillOpacity": 0.0
        },
        tooltip=folium.Tooltip(f"{minutes} min drive"),
        show=True,
    ).add_to(fmap)

# Hospital marker
folium.Marker(
    location=[HOSPITAL_LAT, HOSPITAL_LON],
    tooltip=HOSPITAL_NAME,
    popup=folium.Popup(f"<b>{HOSPITAL_NAME}</b><br>Lovington, NM<br>RN salary: $88,260/yr", max_width=220),
    icon=folium.Icon(color="blue", icon="plus-sign", prefix="glyphicon"),
).add_to(fmap)

# Layer control + colormap
folium.LayerControl(collapsed=False).add_to(fmap)
colormap.add_to(fmap)

# Legend
ring_entries = "".join(
    f'<span style="color:{RING_COLORS[m]};font-weight:bold;">&#9135;&#9135;</span>&nbsp;{m}&nbsp;min&nbsp;&nbsp;'
    for m in sorted(RING_COLORS)
)
stops = ", ".join(GRADIENT_COLORS)
legend_html = f"""
<div style="position:fixed;bottom:30px;right:30px;z-index:1000;background:white;
    padding:14px 18px;border-radius:8px;border:1px solid #ccc;
    font-family:Arial,sans-serif;font-size:12px;
    box-shadow:2px 2px 8px rgba(0,0,0,0.25);min-width:230px;line-height:1.9;">
  <b style="font-size:13px;">Housing Affordability</b><br>
  <span style="color:#555;font-size:11px;">RN salary $88,260/yr &middot; 30% rule &middot; threshold <b>${AFFORD_THRESHOLD:,.0f}</b>/mo</span>
  <div style="margin:8px 0 2px 0;">
    <div style="height:14px;background:linear-gradient(to right,{stops});
        border-radius:3px;border:1px solid #aaa;"></div>
    <div style="display:flex;justify-content:space-between;font-size:10px;color:#444;margin-top:2px;">
      <span>Lower rent<br>(more affordable)</span>
      <span style="text-align:center;">Threshold<br>${AFFORD_THRESHOLD:,.0f}</span>
      <span style="text-align:right;">Higher rent<br>(less affordable)</span>
    </div>
  </div>
  <div style="margin-top:6px;">
    <span style="display:inline-block;width:12px;height:12px;background:{COLOR_NO_DATA};
        border-radius:2px;vertical-align:middle;margin-right:5px;"></span>No rent data
  </div>
  <div style="margin-top:8px;border-top:1px solid #eee;padding-top:6px;">
    <b>Drive-time rings</b><br>{ring_entries}
  </div>
</div>
"""
fmap.get_root().html.add_child(folium.Element(legend_html))

# Save
fmap.save(OUTPUT_HTML)
print(f"Map saved → {OUTPUT_HTML}")

In [ ]:
# ── Cell 12: Preview Map Inline ───────────────────────────────────────────────
# Displays the map directly inside the notebook.
fmap